# Pipeline de Selección de Características EEG
**Proyecto:** PIA — Predicción de Dolor Agudo con EEG  
**Dataset:** `dataset_features_eeg.csv` (5 729 ensayos, 95 sujetos)  
**Grupo:** 4  

---

## Descripción general

Este notebook implementa un pipeline de dos filtros para seleccionar las bandas EEG
más relevantes para predecir el nivel de dolor percibido (escala NRS):

| Celda | Nombre | Propósito |
|-------|--------|-----------|
| `00` | Portada | Descripción del proyecto y tabla de contenidos |
| `01` | Dependencias | Instalación y configuración del entorno |
| `02` | Carga del dataset | Upload interactivo + mapeo NRS + validación |
| `03` | Filtro 1 — Pearson | Eliminación de multicolinealidad (corr > 0.85) |
| `04` | Filtro 2 — MI | Ranking de relevancia con Mutual Information |
| `05` | Visualización | Heatmap de correlación + Ranking VIP |
| `06` | Reporte final | Resumen de variables eliminadas y seleccionadas |

## 01 · Dependencias

Colab ya incluye `numpy`, `pandas`, `matplotlib`, `seaborn` y `scikit-learn`.
La celda siguiente los importa y configura el estilo visual global.
Si alguna librería faltara, descomenta la línea `!pip install`.

**Variables globales que se definen aquí:**
- `UMBRAL_PEARSON` — umbral de correlación máxima permitida entre predictores
- `MAPA_NRS` — diccionario que traduce triggers de hardware a escala NRS 2-4-6-8
- `BANDAS_EEG` — lista de las 9 características de entrada del modelo

In [ ]:
# 01_dependencias.py
# Ejecutar una sola vez al iniciar el entorno.

# !pip install -q seaborn scikit-learn  # descomenta si faltan

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from google.colab import files
from IPython.display import display

# ── Constantes del proyecto ───────────────────────────────────────────────────
UMBRAL_PEARSON: float = 0.85   # correlación máxima permitida entre predictores
RANDOM_STATE:   int   = 42

# Triggers de hardware → escala NRS (Diccionario de Datos, sección 3.2)
MAPA_NRS: dict[int, int] = {
    32: 2,   # Dolor Leve / Umbral Inicial
    33: 2,
    34: 4,   # Dolor Moderado / Soportable
    35: 4,
    36: 6,   # Dolor Intenso / Distractor
    37: 6,
    38: 8,   # Dolor Severo / Límite de Tolerancia
}

BANDAS_EEG: list[str] = [
    "theta_frontal",  "theta_central",  "theta_parietal",
    "alpha_frontal",  "alpha_central",  "alpha_parietal",
    "beta_frontal",   "beta_central",   "beta_parietal",
]

# ── Estilo visual global ──────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

print("  Entorno configurado correctamente.")
print(f"   • Umbral Pearson : {UMBRAL_PEARSON}")
print(f"   • Bandas EEG     : {len(BANDAS_EEG)} características")
print(f"   • Niveles NRS    : {sorted(set(MAPA_NRS.values()))}")

## 02 · Carga del dataset

Se usa `google.colab.files.upload()` para subir el CSV directamente desde el equipo local,
sin necesidad de montar Google Drive ni editar rutas.

**Operaciones que realiza esta celda:**
1. Abre el diálogo de selección de archivo
2. Lee el CSV en memoria con `pandas`
3. Valida que las columnas `nrs_score`, `subject_id` y las 9 bandas EEG existan
4. Traduce los triggers de hardware a escala NRS usando `MAPA_NRS`
5. Muestra una vista previa de 5 filas para confirmar la carga

**Salida esperada:** `df` (DataFrame), `X` (features), `y` (target NRS)

In [ ]:
# 02_carga_dataset.py

def cargar_dataset() -> pd.DataFrame:
    """
    Solicita al usuario subir el CSV desde su equipo local.
    Aplica el mapeo de triggers → NRS y valida columnas requeridas.

    Returns
    -------
    pd.DataFrame
        DataFrame con nrs_score mapeado y listo para análisis.

    Raises
    ------
    ValueError
        Si faltan columnas requeridas o hay triggers no reconocidos.
    """
    print("  Selecciona 'dataset_features_eeg.csv' para subirlo:")
    uploaded = files.upload()

    nombre_archivo = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[nombre_archivo]))

    # Validación de columnas
    columnas_requeridas = {"nrs_score", "subject_id"} | set(BANDAS_EEG)
    faltantes = columnas_requeridas - set(df.columns)
    if faltantes:
        raise ValueError(f"  Columnas no encontradas: {faltantes}")

    # Mapeo de triggers → NRS
    df["nrs_score"] = df["nrs_score"].map(MAPA_NRS)
    if df["nrs_score"].isna().any():
        n_nan = df["nrs_score"].isna().sum()
        raise ValueError(
            f"❌  {n_nan} filas con trigger no reconocido. "
            f"Revisa MAPA_NRS. Valores únicos: {df['nrs_score'].unique()}"
        )

    print(
        f"\n  Dataset cargado exitosamente\n"
        f"   • Ensayos    : {len(df):,}\n"
        f"   • Sujetos    : {df['subject_id'].nunique()}\n"
        f"   • Niveles NRS: {sorted(df['nrs_score'].unique())}"
    )
    return df


df = cargar_dataset()
X  = df[BANDAS_EEG]   # matriz de features (9 bandas EEG)
y  = df["nrs_score"]  # variable objetivo (NRS 2-4-6-8)

print("\n  Vista previa (5 primeras filas):")
display(df[["subject_id", "nrs_score"] + BANDAS_EEG].head())

## 03 · Filtro 1 — Eliminación de Multicolinealidad (Pearson)

Cuando dos predictores están altamente correlacionados entre sí,
aportan información redundante al modelo y pueden distorsionar coeficientes
e inflar la varianza de las estimaciones.

**Algoritmo iterativo:**
```
mientras existan pares con |r| > UMBRAL_PEARSON:
    1. Calcular matriz de correlación absoluta actual
    2. Identificar todos los pares problemáticos
    3. Calcular el promedio de correlación de cada variable involucrada
    4. Eliminar la variable con mayor promedio (la más redundante globalmente)
```

**Por qué este criterio?**  
Eliminar la variable más correlacionada *en promedio* con todas las demás
minimiza la pérdida de información al remover la característica más sustituible.

**Salida:** `X_filtrado`, `eliminadas`, `matriz_corr` (original, para heatmap)

In [ ]:
# 03_filtro_pearson.py

def eliminar_multicolinealidad(
    X: pd.DataFrame,
    umbral: float = UMBRAL_PEARSON,
) -> tuple[pd.DataFrame, list[str], pd.DataFrame]:
    """
    Elimina predictores redundantes mediante correlación de Pearson absoluta.

    Parameters
    ----------
    X      : DataFrame con las características de entrada.
    umbral : Umbral máximo de correlación permitida (default 0.85).

    Returns
    -------
    X_filtrado   : DataFrame sin columnas redundantes.
    eliminadas   : Lista de nombres de columnas eliminadas.
    matriz_corr  : Matriz de correlación absoluta ORIGINAL.
    """
    matriz_corr = X.corr().abs()   # guardamos la original para el heatmap
    X_filtrado  = X.copy()
    eliminadas: list[str] = []
    iteracion = 1

    while True:
        corr_actual = X_filtrado.corr().abs()
        mascara_sup = np.triu(np.ones(corr_actual.shape), k=1).astype(bool)
        sup_actual  = corr_actual.where(mascara_sup)

        # Detectar pares con correlación sobre el umbral
        pares_altos = [
            (col, fila)
            for col in sup_actual.columns
            for fila in sup_actual.index
            if sup_actual.loc[fila, col] > umbral
        ]

        if not pares_altos:
            print(f"   Iteración {iteracion}: sin pares redundantes → proceso finalizado.")
            break

        # Elegir la variable con mayor promedio de correlación (más redundante)
        involucradas  = {v for par in pares_altos for v in par}
        promedio_corr = {var: corr_actual[var].drop(var).mean() for var in involucradas}
        a_eliminar    = max(promedio_corr, key=promedio_corr.get)

        print(
            f"   Iteración {iteracion}: {len(pares_altos)} par(es) > {umbral} → "
            f"eliminando '{a_eliminar}' (corr. media = {promedio_corr[a_eliminar]:.4f})"
        )

        X_filtrado = X_filtrado.drop(columns=[a_eliminar])
        eliminadas.append(a_eliminar)
        iteracion += 1

    return X_filtrado, eliminadas, matriz_corr


print(f"🔎  Filtro 1 — Multicolinealidad (umbral Pearson = {UMBRAL_PEARSON})\n")
X_filtrado, eliminadas, matriz_corr = eliminar_multicolinealidad(X)

print(f"\n📊  Resultado Filtro 1:")
print(f"   • Variables originales : {len(BANDAS_EEG)}")
print(f"   • Eliminadas           : {len(eliminadas)}  →  {eliminadas or 'ninguna'}")
print(f"   • Supervivientes       : {len(X_filtrado.columns)}  →  {X_filtrado.columns.tolist()}")

## 04 · Filtro 2 — Ranking por Mutual Information

La **Información Mutua (MI)** mide cuánta información comparte una variable
con la variable objetivo, capturando relaciones tanto lineales como no lineales.
Un puntaje MI = 0 indica independencia estadística; valores más altos indican
mayor relevancia predictiva.

$$I(X;Y) = \sum_{x,y} P(x,y) \log \frac{P(x,y)}{P(x)P(y)}$$

**Implementación:** `sklearn.feature_selection.mutual_info_classif`  
**Entrada:** `X_filtrado` (solo características supervivientes del Filtro 1)  
**Salida:** `df_mi` — DataFrame con ranking de mayor a menor relevancia

In [ ]:
# 04_filtro_mutual_information.py

def calcular_mutual_information(
    X_filtrado: pd.DataFrame,
    y: pd.Series,
) -> pd.DataFrame:
    """
    Calcula el puntaje de Mutual Information de cada característica
    superviviente respecto a la variable objetivo `nrs_score`.

    Parameters
    ----------
    X_filtrado : DataFrame con características post Filtro 1.
    y          : Serie con los valores NRS (variable objetivo).

    Returns
    -------
    pd.DataFrame
        Ranking ordenado de mayor a menor puntaje MI, índice desde 1.
    """
    scores_mi = mutual_info_classif(X_filtrado, y, random_state=RANDOM_STATE)

    df_mi = (
        pd.DataFrame({"Onda_Cerebral": X_filtrado.columns, "Puntaje_MI": scores_mi})
        .sort_values("Puntaje_MI", ascending=False)
        .reset_index(drop=True)
    )
    df_mi.index += 1   # ranking legible desde 1
    return df_mi


print("  Filtro 2 — Mutual Information vs. nrs_score\n")
df_mi = calcular_mutual_information(X_filtrado, y)

print("  Ranking VIP completo:")
display(
    df_mi.style
    .background_gradient(subset=["Puntaje_MI"], cmap="viridis")
    .format({"Puntaje_MI": "{:.6f}"})
    .set_caption("Ranking MI — Características supervivientes del Filtro 1")
)

## 05 · Visualización — Heatmap + Ranking VIP

Se generan dos paneles complementarios:

| Panel | Contenido | Para qué sirve |
|-------|-----------|----------------|
| **Izquierdo** | Heatmap de correlación original (9×9) | Identificar visualmente qué pares superaron el umbral (borde rojo) |
| **Derecho** | Barras horizontales con puntaje MI | Comparar la relevancia relativa de las características supervivientes |

> El gráfico se guarda como `eeg_feature_selection.png` y se descarga automáticamente.  
> **Antes de hacer commit a GitHub**, elimina la salida de esta celda para no subir imágenes binarias al repo.

In [ ]:
# 05_visualizacion.py

def visualizar_resultados(
    matriz_corr_original: pd.DataFrame,
    X_filtrado: pd.DataFrame,
    eliminadas: list[str],
    df_mi: pd.DataFrame,
) -> None:
    """
    Genera heatmap de correlación + ranking VIP de Mutual Information.
    Guarda el resultado como PNG y lo descarga desde Colab.

    Parameters
    ----------
    matriz_corr_original : Matriz de correlación ANTES del Filtro 1.
    X_filtrado           : DataFrame post Filtro 1 (para referencia de supervivientes).
    eliminadas           : Variables eliminadas (usadas en el título del heatmap).
    df_mi                : Ranking de Mutual Information del Filtro 2.
    """
    fig = plt.figure(figsize=(18, 7))
    fig.suptitle(
        "Pipeline de Selección de Características EEG\n"
        "Filtro Pearson (multicolinealidad) + Ranking Mutual Information",
        fontsize=14, fontweight="bold", y=1.02,
    )
    gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.38)

    # ── Panel izquierdo: Heatmap de correlación original ─────────────────────
    ax_heat = fig.add_subplot(gs[0, 0])
    mascara_diag = np.eye(len(matriz_corr_original), dtype=bool)

    sns.heatmap(
        matriz_corr_original,
        ax=ax_heat,
        annot=matriz_corr_original.round(2).astype(str),
        fmt="",
        cmap="RdYlGn_r",
        vmin=0, vmax=1,
        linewidths=0.5,
        linecolor="white",
        square=True,
        cbar_kws={"label": "Correlación de Pearson (abs.)"},
        mask=mascara_diag,
    )

    # Marcar en rojo los pares que superaron el umbral
    n = len(matriz_corr_original.columns)
    for i in range(n):
        for j in range(n):
            if i != j and matriz_corr_original.iloc[i, j] > UMBRAL_PEARSON:
                ax_heat.add_patch(
                    plt.Rectangle((j, i), 1, 1, fill=False, edgecolor="red", lw=2.5)
                )

    ax_heat.set_title(
        f"Correlaciones entre Bandas EEG\n(borde rojo = correlación > {UMBRAL_PEARSON})",
        fontsize=11,
    )
    ax_heat.tick_params(axis="x", rotation=45)
    ax_heat.tick_params(axis="y", rotation=0)

    # ── Panel derecho: Ranking VIP ────────────────────────────────────────────
    ax_vip = fig.add_subplot(gs[0, 1])
    colores_vip = sns.color_palette("viridis", len(df_mi))

    bars = ax_vip.barh(
        df_mi["Onda_Cerebral"][::-1],
        df_mi["Puntaje_MI"][::-1],
        color=colores_vip,
        edgecolor="white",
        height=0.65,
    )

    for bar, val in zip(bars, df_mi["Puntaje_MI"][::-1]):
        ax_vip.text(
            bar.get_width() + 0.0003,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}",
            va="center", ha="left", fontsize=9,
        )

    ax_vip.set_title(
        "Ranking VIP — Importancia por Mutual Information\n"
        "(solo características sin multicolinealidad)",
        fontsize=11,
    )
    ax_vip.set_xlabel("Puntaje de Información Mutua (Mayor = Más Relevante)")
    ax_vip.set_ylabel("Zona y Banda EEG")
    ax_vip.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()

    nombre_img = "eeg_feature_selection.png"
    plt.savefig(nombre_img, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"\n💾  Gráfico guardado: '{nombre_img}'")
    files.download(nombre_img)   # descarga automática en Colab


visualizar_resultados(matriz_corr, X_filtrado, eliminadas, df_mi)